In [20]:
import os
import joblib
import torch
import torch.nn as nn

## Load Pre-trained Models

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SPAM_MODEL_PATH = "../models/nn_saved_model.pth"
SPAM_VECTORIZER_PATH = "../models/nn_spam_vectorizer.joblib"

PHISH_MODEL_PATH = "../models/phishing_nn_model.pth"
PHISH_VECTORIZER_PATH = "../models/phishing_nn_vectorizer.joblib"

In [22]:
class FeedForwardNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout_p=0.3):
        super().__init__()
        self.neural_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x):
        return self.neural_net(x)

In [23]:
def load_nn_model(model_path, vectorizer_path):
    # Load vectorizer
    vectorizer = joblib.load(vectorizer_path)
    
    # Build model with correct input_dim
    input_dim = len(vectorizer.get_feature_names_out())
    model = FeedForwardNet(input_dim=input_dim).to(device)
    
    # Load weights
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    
    return model, vectorizer

spam_model, spam_vectorizer = load_nn_model(SPAM_MODEL_PATH, SPAM_VECTORIZER_PATH)
phish_model, phish_vectorizer = load_nn_model(PHISH_MODEL_PATH, PHISH_VECTORIZER_PATH)

## Test the Pipeline with Examples

In [24]:
def classify_email_nn(
    email_text: str,
    spam_threshold: float = 0.5,
    phish_threshold: float = 0.5,
):
    # --- Stage 1: Spam vs Ham ---
    X_spam = spam_vectorizer.transform([email_text])
    X_spam = torch.tensor(X_spam.toarray(), dtype=torch.float32).to(device)

    with torch.no_grad():
        spam_logits = spam_model(X_spam)
        spam_prob = torch.sigmoid(spam_logits).item()

    is_spam = spam_prob >= spam_threshold

    result = {
        "email_text": email_text,
        "spam_probability": spam_prob,
        "is_spam": is_spam,
        "phish_probability": None,
        "is_phish": None,
        "final_label": None,
    }

    # If not spam, stop here.
    if not is_spam:
        result["final_label"] = "Ham / Not Phishing"
        return result

    # --- Stage 2: Phish vs Not Phish (only for spammy emails) ---
    X_phish = phish_vectorizer.transform([email_text])
    X_phish = torch.tensor(X_phish.toarray(), dtype=torch.float32).to(device)

    with torch.no_grad():
        phish_logits = phish_model(X_phish)
        phish_prob = torch.sigmoid(phish_logits).item()

    is_phish = phish_prob >= phish_threshold

    result["phish_probability"] = phish_prob
    result["is_phish"] = is_phish
    result["final_label"] = "Phishing" if is_phish else "Non-phishing Spam"

    return result


## Batch Classification

In [25]:
def display_result_nn(result):
    print("=" * 60)
    print("EMAIL PREVIEW:\n")
    print(result["email_text"][:500], "\n")
    
    print(f"Stage 1 - Spam Probability: {result['spam_probability']:.3f}")
    print(f"Classified as Spam: {result['is_spam']}")
    
    if result["phish_probability"] is not None:
        print(f"\nStage 2 - Phish Probability: {result['phish_probability']:.3f}")
        print(f"Classified as Phishing: {result['is_phish']}")
    
    print(f"\nFINAL LABEL: {result['final_label']}")
    print("=" * 60)


## Interactive Classification (Enter Your Own Email)

In [27]:
test_email = """
URGENT: Your account has been compromised!

We noticed suspicious login attempts. Click the link below to verify
your credentials and secure your account:

[Verify Now]

If you do not act within 24 hours, your account will be locked.
"""

res = classify_email_nn(test_email)
display_result_nn(res)

EMAIL PREVIEW:


URGENT: Your account has been compromised!

We noticed suspicious login attempts. Click the link below to verify
your credentials and secure your account:

[Verify Now]

If you do not act within 24 hours, your account will be locked.
 

Stage 1 - Spam Probability: 1.000
Classified as Spam: True

Stage 2 - Phish Probability: 0.999
Classified as Phishing: True

FINAL LABEL: Phishing


In [29]:
test_email = """
Dear Hiring Manager,

I'm writing to express my interest in the [job title] position advertised on your company's website. Please find my cover letter and résumé attached below. I'm excited to contribute my skills and experience to your team.

[Brief cover letter content highlighting relevant qualifications.]

Thank you for considering my application. I'd love to talk with you more about the position.

Sincerely,

[Your full name]
"""

res = classify_email_nn(test_email)
display_result_nn(res)

EMAIL PREVIEW:


Dear Hiring Manager,

I'm writing to express my interest in the [job title] position advertised on your company's website. Please find my cover letter and résumé attached below. I'm excited to contribute my skills and experience to your team.

[Brief cover letter content highlighting relevant qualifications.]

Thank you for considering my application. I'd love to talk with you more about the position.

Sincerely,

[Your full name]
 

Stage 1 - Spam Probability: 0.000
Classified as Spam: False

FINAL LABEL: Ham / Not Phishing
